# 01 - Data Preparation (Kaggle Credit Card Transactions)

This notebook implements Phase 1 (Data Preparation) for:

- Dataset: https://www.kaggle.com/datasets/priyamchoksi/credit-card-transactions-dataset
- Goals: ingest raw CSV in Spark, clean invalid/missing/duplicate records, standardize schema, and write Parquet for reuse in later phases.

It is designed to run in both:

- Local environment (from project root)
- Google Colab (after cloning repo)

In [11]:
# If needed (Colab/local), install dependencies:
# !pip install -r requirements.txt

from pathlib import Path
import sys

# Ensure project root is importable whether notebook runs from repo root or notebooks/.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    LongType,
)

from src.pipeline import (
    create_spark_session,
    normalize_column_names,
    standardize_schema,
    clean_transactions,
)

print(f"Imports loaded. Project root: {project_root}")

Imports loaded. Project root: /home/elaf/Desktop/big-data-analytics-credit-card-transactions


## Dataset download (Kaggle)

Use one of the following methods:

1. **Manual download** from Kaggle URL and place CSV in `data/raw/`
2. **Kaggle API** (requires `kaggle.json` credentials)

Example API command (run in terminal/Colab):

```bash
kaggle datasets download -d priyamchoksi/credit-card-transactions-dataset -p data/raw --unzip
```

After download, ensure at least one CSV file exists in `data/raw/`.

In [12]:
# Resolve project root safely from either repo root or notebooks/ execution context.
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        f"Could not locate project root from current directory: {cwd}"
    )

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

csv_candidates = sorted(raw_dir.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(
        f"No CSV files found in: {raw_dir}. Download dataset and place CSV there."
    )

input_csv = csv_candidates[0]
output_parquet = processed_dir / "transactions_parquet"

print(f"Current working directory: {cwd}")
print(f"Resolved project root: {project_root}")
print(f"Using input CSV: {input_csv}")
print(f"Output Parquet path: {output_parquet}")

Current working directory: /home/elaf/Desktop/big-data-analytics-credit-card-transactions/notebooks
Resolved project root: /home/elaf/Desktop/big-data-analytics-credit-card-transactions
Using input CSV: /home/elaf/Desktop/big-data-analytics-credit-card-transactions/data/raw/credit_card_transactions.csv
Output Parquet path: /home/elaf/Desktop/big-data-analytics-credit-card-transactions/data/processed/transactions_parquet


In [14]:
# Explicit schema for common columns in this dataset family.
# Unknown or additional columns can still be handled by schema normalization.
raw_schema = StructType([
    StructField("Unnamed: 0", LongType(), True),
    StructField("trans_date_trans_time", StringType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("city_pop", LongType(), True),
    StructField("job", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", LongType(), True),
    StructField("merch_lat", DoubleType(), True),
    StructField("merch_long", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),
    StructField("merch_zipcode", StringType(), True),
])

spark = create_spark_session(app_name="CreditCardDataPrepNotebook")

raw_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "DROPMALFORMED")
    .schema(raw_schema)
    .load(str(input_csv))
)

if "Unnamed: 0" in raw_df.columns:
    raw_df = raw_df.drop("Unnamed: 0")

print("Raw row count:", raw_df.count())
raw_df.printSchema()
raw_df.show(5, truncate=False)

Raw row count: 1296675
root
 |-- trans_date_trans_time: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: long (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_zipcode: string (nullable = true)

+---------------------+----------------+----------------------------------+-----

In [15]:
normalized_df = normalize_column_names(raw_df)
standardized_df = standardize_schema(normalized_df)
cleaned_df = clean_transactions(standardized_df)

print("Cleaned row count:", cleaned_df.count())
cleaned_df.printSchema()
cleaned_df.select("transaction_id", "transaction_ts", "amount", "is_fraud").show(10, truncate=False)

Cleaned row count: 1296675
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)
 |-- card_id: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: long (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- merch_zipcode: string (nullable = true)



+--------------------------------+-------------------+------+--------+
|transaction_id                  |transaction_ts     |amount|is_fraud|
+--------------------------------+-------------------+------+--------+
|88fde595524ba43924f36bd397b86413|2019-01-01 00:40:44|212.45|0       |
|70ca7fe41f09770d724a027152dcb9f0|2019-01-01 00:46:18|2.76  |0       |
|fdce65baf3dd8e897f03fb09f6c78b9c|2019-01-01 02:35:45|2.07  |0       |
|0d0cbcb30d4a99512542a84eb09469bb|2019-01-01 07:17:04|16.08 |0       |
|fd9ecf577698e23eafc204f2b44132c6|2019-01-01 13:36:24|8.02  |0       |
|3b3097e61b0cbcbba59e03db6ca01038|2019-01-01 21:17:07|95.82 |0       |
|74089fa10cbe76f3f91194bf32f3711c|2019-01-01 22:35:15|24.01 |0       |
|a4325187ce805cdabd578069312fccfc|2019-01-02 01:14:35|43.56 |0       |
|ea28ab0a0e0a58cdd94e7abf0d018d43|2019-01-02 10:59:35|103.66|0       |
|2d930357e58322e0e3cd5bc19cdbb3b5|2019-01-02 12:45:50|60.75 |0       |
+--------------------------------+-------------------+------+--------+
only s

In [16]:
(
    cleaned_df.write
    .mode("overwrite")
    .parquet(str(output_parquet))
)

print(f"Parquet written to: {output_parquet}")

# Lightweight quality checks
print("Null amounts:", cleaned_df.filter(cleaned_df.amount.isNull()).count() if "amount" in cleaned_df.columns else "N/A")
print("Duplicate transaction_ids:", cleaned_df.groupBy("transaction_id").count().filter("count > 1").count())

spark.stop()
print("Spark session stopped.")

26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/28 08:32:27 WARN MemoryManager: Total allocation exceeds 95.

Parquet written to: /home/elaf/Desktop/big-data-analytics-credit-card-transactions/data/processed/transactions_parquet


Null amounts: 0


Duplicate transaction_ids: 0
Spark session stopped.
